# Huấn luyện YOLO11m Hybrid (VinDr + NIH) cho CheXNet — Google Colab (A100 40GB)

Notebook này huấn luyện **YOLO11m** để phát hiện tổn thương trên X-quang ngực,
sử dụng bộ dữ liệu **Hybrid** kết hợp:
- **VinDr-CXR**: BBox chuẩn từ bác sĩ (10 bệnh có bbox)
- **NIH ChestX-ray14**: BBox sinh tự động (Pseudo-labels) từ **CheXNet Hybrid Model** cho 4 bệnh VinDr không có

### Các cải tiến so với bản trước:
1. **Knowledge Distillation** — CheXNet Attention Map sinh Pseudo BBox cho Pneumonia, Edema, Emphysema, Hernia
2. **Dữ liệu Hybrid** — VinDr (~18K) + NIH (~112K ảnh), YOLO học từ cả 2 bộ
3. **WBF trên Ground Truth** — Gộp boxes từ nhiều bác sĩ (VinDr) thành consensus box
4. **Progressive Training** — Phase 1 (640px) rồi Phase 2 (1024px)
5. **TTA + WBF Inference** — Dự đoán ảnh gốc + flip, gộp WBF
6. **Cascade 2 giai đoạn** — `Final_Score = P_classifier * Conf_YOLO`
7. **14 classes đồng bộ CheXNet** — Mapping trực tiếp với CheXNet classifier

### Chuẩn bị trên Colab:
1. Bật **GPU A100** (Runtime > Change runtime type > A100 GPU)
2. Mount **Google Drive** chứa dữ liệu `Database_512` (data_1.zip..data_10.zip)
3. Upload `hybrid_model_sota.pth` lên Drive tại `CheXNet/`

### Tối ưu cho A100 40GB:
- **Phase 1** (640px): batch=**80**, workers=8, cache='ram'
- **Phase 2** (1024px): batch=**28**, workers=8, cache='ram'
- **bf16** thay vì fp16 (A100 hỗ trợ native BFloat16)
- **Cache ảnh vào RAM** — A100 Colab có 83GB RAM, đủ cache toàn bộ dataset

---
## 0. Chuẩn bị dữ liệu (Clone repo + Giải nén + Gom ảnh)

3 cell dưới đây sẽ:
1. Clone repo CheXNet và mount Google Drive
2. Tự động tìm và giải nén tất cả file `.zip` từ Drive vào `/content/CheXNet/Database`
3. Gom tất cả ảnh từ các thư mục con ra thư mục gốc `Database`

In [ ]:
# =============================================
# CELL 1: Clone repo CheXNet + Mount Drive
# =============================================
!git clone -b test https://github.com/lecuong2512/CheXNet

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# =============================================
# CELL 2: Giải nén dữ liệu từ Google Drive
# =============================================
import os
import shutil
import zipfile
import time
import glob
import re
from pathlib import Path

def natural_sort_key(s):
    return [int(text) if text.isdigit() else text.lower()
            for text in re.split('([0-9]+)', s)]

SOURCE_DIR = "/content/drive/MyDrive/Database_512"
TARGET_DIR = "/content/CheXNet/Database"

os.makedirs(TARGET_DIR, exist_ok=True)

print("🚀 Đang quét tìm các tệp .zip trong thư mục nguồn...")

search_pattern = os.path.join(SOURCE_DIR, "*.zip")
zip_files = glob.glob(search_pattern)
zip_files.sort(key=natural_sort_key)

total_files = len(zip_files)

if total_files == 0:
    print(f"⚠️  Không tìm thấy tệp .zip nào trong: {SOURCE_DIR}")
else:
    print(f"✅ Đã tìm thấy {total_files} tệp zip.")
    print("-" * 50)

    for i, source_path in enumerate(zip_files, 1):
        file_name = os.path.basename(source_path)
        temp_file = os.path.join("/content", file_name)

        print(f"📦 [{i}/{total_files}] Đang xử lý: {file_name}")

        file_size = os.path.getsize(source_path)
        print(f"   📥 Đang sao chép... ({file_size / 1024 / 1024:.1f} MB)")
        start_time = time.time()

        BUFFER_SIZE = 1024 * 1024
        copied = 0

        try:
            with open(source_path, 'rb') as src, open(temp_file, 'wb') as dst:
                while True:
                    buffer = src.read(BUFFER_SIZE)
                    if not buffer:
                        break
                    dst.write(buffer)
                    copied += len(buffer)

                    duration = time.time() - start_time
                    if duration > 0:
                        speed = copied / 1024 / 1024 / duration
                        percent = copied * 100 / file_size
                        print(f"\r   ⏳ Copy: {percent:5.1f}% | {speed:6.2f} MB/s", end='', flush=True)

            print()

            print(f"   📂 Đang giải nén vào đích...")
            unzip_start = time.time()

            with zipfile.ZipFile(temp_file, 'r') as zip_ref:
                file_list = zip_ref.namelist()
                total_items = len(file_list)

                for idx, file in enumerate(file_list, 1):
                    zip_ref.extract(file, TARGET_DIR)
                    if idx % 50 == 0 or idx == total_items:
                        percent = idx * 100 / total_items
                        print(f"\r   🔨 Unzip: {percent:5.1f}% ({idx}/{total_items} files)", end='')

            unzip_duration = time.time() - unzip_start
            print(f"\n   ✅ Giải nén xong trong {unzip_duration:.1f}s")

            os.remove(temp_file)
            print(f"   🗑️  Đã xóa file tạm.")
            print("-" * 50)

        except Exception as e:
            print(f"\n❌ LỖI khi xử lý file {file_name}: {e}")
            continue

print(f"\n🎉 HOÀN TẤT! Đã xử lý xong {total_files} file.")

In [ ]:
# =============================================
# CELL 3: Gom tất cả ảnh ra thư mục gốc Database
# =============================================
import os, shutil

root_dir = '/content/CheXNet/Database'
target_dir = '/content/CheXNet/Database'

exts = ('.jpg', '.jpeg', '.png', '.bmp', '.dicom', '.dcm')

count = 0
duplicates = 0
for subdir, _, files in os.walk(root_dir):
    for f in files:
        if f.lower().endswith(exts):
            src = os.path.join(subdir, f)
            dst = os.path.join(target_dir, f)

            if src == dst:
                continue

            if os.path.exists(dst):
                duplicates += 1
                continue  # bo qua file trung ten

            shutil.move(src, dst)
            count += 1

print(f"✅ Đã di chuyển {count} ảnh ra ngoài {target_dir}")
if duplicates > 0:
    print(f"⚠️  Bỏ qua {duplicates} file trùng tên")

---
## 1. Cài đặt thư viện & Cấu hình đường dẫn

In [ ]:
!pip install -q ultralytics ensemble-boxes scikit-learn timm

import os
import sys
import cv2
import glob
import shutil
import gc
import numpy as np
import pandas as pd
from collections import Counter, defaultdict
from tqdm.auto import tqdm
from ensemble_boxes import weighted_boxes_fusion
from ultralytics import YOLO
import torch
import torch.nn.functional as F
import torchvision.transforms as transforms
from PIL import Image

# ======================================================
# CAU HINH DUONG DAN (CHO GOOGLE COLAB + A100 40GB)
# ======================================================

# Dataset anh VinDr + NIH (da gom PHANG ra Database)
DATA_ROOT = '/content/CheXNet/Database'

# CSV files (nam trong Dataset/, KHONG phai Database/)
TRAIN_CSV = '/content/CheXNet/Dataset/train_list.csv'
VAL_CSV   = '/content/CheXNet/Dataset/val_list.csv'
TEST_CSV  = '/content/CheXNet/Dataset/test_list.csv'

# CheXNet Model code (Model.py, checkpoint_utils.py)
CHEXNET_CODE_DIR   = '/content/CheXNet/Models'
# CheXNet Model weights
CHEXNET_MODEL_PATH = '/content/drive/MyDrive/CheXNet/hybrid_model_sota.pth'

# Thu muc lam viec (output)
WORK_DIR = '/content/yolo_hybrid'
os.makedirs(WORK_DIR, exist_ok=True)

# Detect GPU
n_gpus = torch.cuda.device_count()
for i in range(n_gpus):
    props = torch.cuda.get_device_properties(i)
    mem = getattr(props, 'total_memory', None) or getattr(props, 'total_mem', 0)
    print(f'GPU {i}: {torch.cuda.get_device_name(i)} ({mem / 1e9:.1f} GB)')
DEVICE = list(range(n_gpus)) if n_gpus > 1 else 0
GPU_DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'YOLO device: {DEVICE}  |  CheXNet device: {GPU_DEVICE}')

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print(f'\n🔥 GPU: {gpu_name}')
    if 'A100' in gpu_name:
        print('✅ A100 detected — sẽ dùng bf16 + batch lớn')
    print(f'BF16 support: {torch.cuda.is_bf16_supported()}')

## 2. Ánh xạ 14 Class YOLO - CheXNet

YOLO sẽ có **14 classes** (bỏ `No Finding`), đồng bộ **1-1** với CheXNet classifier:

| YOLO ID | Tên bệnh | Nguồn BBox | CheXNet idx |
|---------|-----------|------------|-------------|
| 0 | Atelectasis | VinDr GT | 1 |
| 1 | Cardiomegaly | VinDr GT | 2 |
| 2 | Effusion | VinDr GT | 3 |
| 3 | Infiltration | VinDr GT | 4 |
| 4 | Mass | VinDr GT | 5 |
| 5 | Nodule | VinDr GT | 6 |
| 6 | **Pneumonia** | **CheXNet Pseudo** | 7 |
| 7 | Pneumothorax | VinDr GT | 8 |
| 8 | Consolidation | VinDr GT | 9 |
| 9 | **Edema** | **CheXNet Pseudo** | 10 |
| 10 | **Emphysema** | **CheXNet Pseudo** | 11 |
| 11 | Fibrosis | VinDr GT | 12 |
| 12 | Pleural_Thickening | VinDr GT | 13 |
| 13 | **Hernia** | **CheXNet Pseudo** | 14 |

> 4 bệnh in đậm: VinDr không có BBox, dùng CheXNet Attention Map sinh Pseudo-labels từ ảnh NIH.

In [ ]:
# -- 14 YOLO Classes (dong bo CheXNet, bo No Finding) --
YOLO_CLASSES = [
    'Atelectasis',        # 0
    'Cardiomegaly',       # 1
    'Effusion',           # 2
    'Infiltration',       # 3
    'Mass',               # 4
    'Nodule',             # 5
    'Pneumonia',          # 6  <- Pseudo from NIH
    'Pneumothorax',       # 7
    'Consolidation',      # 8
    'Edema',              # 9  <- Pseudo from NIH
    'Emphysema',          # 10 <- Pseudo from NIH
    'Fibrosis',           # 11
    'Pleural_Thickening', # 12
    'Hernia',             # 13 <- Pseudo from NIH
]

DISEASE_TO_YOLO_ID = {name: i for i, name in enumerate(YOLO_CLASSES)}

PSEUDO_DISEASES = ['Pneumonia', 'Edema', 'Emphysema', 'Hernia']
PSEUDO_YOLO_IDS = {d: DISEASE_TO_YOLO_ID[d] for d in PSEUDO_DISEASES}

CHEXNET_CLASS_NAMES = [
    'No Finding', 'Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltration',
    'Mass', 'Nodule', 'Pneumonia', 'Pneumothorax', 'Consolidation',
    'Edema', 'Emphysema', 'Fibrosis', 'Pleural_Thickening', 'Hernia'
]
PSEUDO_CHEXNET_IDX = {d: CHEXNET_CLASS_NAMES.index(d) for d in PSEUDO_DISEASES}
print(f'Pseudo disease -> CheXNet channel: {PSEUDO_CHEXNET_IDX}')

# -- Doc CSV --
df_train = pd.read_csv(TRAIN_CSV)
df_val   = pd.read_csv(VAL_CSV)
print(f'\nTrain: {len(df_train)} rows, unique images: {df_train["Image Index"].nunique()}')
print(f'Val:   {len(df_val)} rows, unique images: {df_val["Image Index"].nunique()}')

for split_name, df in [('Train', df_train), ('Val', df_val)]:
    print(f'\n{split_name} source distribution:')
    print(df.groupby('Source')['Image Index'].nunique())

# -- Cache kich thuoc anh vao RAM (A100 Colab co 83GB RAM) --
# Cell 3 da gom TAT CA anh vao DATA_ROOT (flat, khong con thu muc con)
print('\n🚀 Caching image sizes into RAM...')
IMG_EXTENSIONS = {'.png', '.jpg', '.jpeg'}
image_size_cache = {}
skipped = 0
all_images = [f for f in os.listdir(DATA_ROOT) if os.path.splitext(f)[1].lower() in IMG_EXTENSIONS]
for f in tqdm(all_images, desc='Caching sizes'):
    try:
        with Image.open(os.path.join(DATA_ROOT, f)) as img:
            image_size_cache[f] = img.size  # (width, height)
    except Exception:
        skipped += 1
print(f'✅ Cached {len(image_size_cache):,} image sizes ({skipped} skipped)')

# Kiem tra nhanh
for split_name, df in [('Train', df_train), ('Val', df_val)]:
    filenames = df['Image Index'].unique()
    found = sum(1 for f in filenames if f in image_size_cache)
    missing = len(filenames) - found
    print(f'{split_name}: {found}/{len(filenames)} found, {missing} missing')
    if missing > 0:
        missing_samples = [f for f in filenames if f not in image_size_cache][:5]
        print(f'  Sample missing: {missing_samples}')

## 3. Hợp nhất Ground Truth bằng WBF (VinDr)

Trong VinDr-CXR, **mỗi ảnh được nhiều bác sĩ** (cột `rad_id`) đánh dấu độc lập.
Dùng **Weighted Boxes Fusion (WBF)** gộp các boxes trùng lặp thành 1 consensus box.

In [ ]:
def fuse_vindr_boxes(group_df, img_w, img_h):
    """Gop boxes tu nhieu bac si cho 1 anh VinDr bang WBF."""
    has_bbox = group_df['x_min'].notna() & group_df['x_max'].notna()
    bbox_df = group_df[has_bbox].copy()
    
    if len(bbox_df) == 0:
        return []
    
    rads = bbox_df['rad_id'].dropna().unique()
    
    if len(rads) <= 1:
        results = []
        for _, row in bbox_df.iterrows():
            disease = row['Finding Labels']
            if disease not in DISEASE_TO_YOLO_ID:
                continue
            results.append({
                'yolo_class_id': DISEASE_TO_YOLO_ID[disease],
                'x_min': float(row['x_min']), 'y_min': float(row['y_min']),
                'x_max': float(row['x_max']), 'y_max': float(row['y_max']),
            })
        return results
    
    boxes_list, scores_list, labels_list = [], [], []
    for rad_id in rads:
        rad_df = bbox_df[bbox_df['rad_id'] == rad_id]
        boxes, scores, labels = [], [], []
        for _, row in rad_df.iterrows():
            disease = row['Finding Labels']
            if disease not in DISEASE_TO_YOLO_ID:
                continue
            x1 = np.clip(float(row['x_min']) / img_w, 0, 1)
            y1 = np.clip(float(row['y_min']) / img_h, 0, 1)
            x2 = np.clip(float(row['x_max']) / img_w, 0, 1)
            y2 = np.clip(float(row['y_max']) / img_h, 0, 1)
            if x2 <= x1 or y2 <= y1:
                continue
            boxes.append([x1, y1, x2, y2])
            scores.append(1.0)
            labels.append(DISEASE_TO_YOLO_ID[disease])
        if boxes:
            boxes_list.append(boxes)
            scores_list.append(scores)
            labels_list.append(labels)
    
    if not boxes_list:
        return []
    
    fused_boxes, fused_scores, fused_labels = weighted_boxes_fusion(
        boxes_list, scores_list, labels_list,
        weights=[1.0] * len(boxes_list), iou_thr=0.5, skip_box_thr=0.0001
    )
    results = []
    for box, label in zip(fused_boxes, fused_labels):
        results.append({
            'yolo_class_id': int(label),
            'x_min': box[0] * img_w, 'y_min': box[1] * img_h,
            'x_max': box[2] * img_w, 'y_max': box[3] * img_h,
        })
    return results

print('WBF function ready.')

## 4. Knowledge Distillation: CheXNet sinh Pseudo-Labels bằng Grad-CAM

### Vấn đề
VinDr-CXR **không có BBox** cho 4 bệnh: Pneumonia, Edema, Emphysema, Hernia.
Attention maps (output từ attention head) **gần = 0** trên ảnh NIH vì chúng chỉ
được train bằng Dice Loss trên ảnh VinDr có bbox ground truth.

### Giải pháp: Grad-CAM
Dùng **Grad-CAM** trên `fpn_features` (layer cuối trước pooling):
1. Forward pass qua CheXNet → lấy feature maps từ FPN merge layer
2. Backprop từ **logit của bệnh target** → lấy gradients
3. **Grad-CAM = ReLU(sum(weights × features))** — weights = mean(gradients)
4. Threshold (Otsu / percentile) → findContours → sinh BBox
5. Lọc bbox theo diện tích (> 0.1% ảnh)

> **Tại sao Grad-CAM hoạt động?** Vì classification head **ĐÃ được train trên NIH**
> (BCE loss trên tất cả 15 bệnh). Gradients phản ánh vùng nào trên ảnh đóng góp
> nhiều nhất vào dự đoán bệnh → đó chính là vùng tổn thương.

In [ ]:
# ======================================
# BUOC 3.1: Load CheXNet Hybrid Model
# ======================================
if CHEXNET_CODE_DIR not in sys.path:
    sys.path.insert(0, CHEXNET_CODE_DIR)

from Model import HybridCNNViTModel
from checkpoint_utils import load_checkpoint_safe, extract_state_dict

print('Loading CheXNet Hybrid Model...')
ckpt = load_checkpoint_safe(CHEXNET_MODEL_PATH, device=torch.device('cpu'))
model_size = ckpt.get('model_size', 'base')
img_size = ckpt.get('img_size', 384)
cleaned_sd = extract_state_dict(ckpt)

chexnet_model = HybridCNNViTModel(num_classes=15, model_size=model_size,
                                   img_size=img_size, pretrained=False)
chexnet_model.load_state_dict(cleaned_sd, strict=False)
chexnet_model = chexnet_model.to(GPU_DEVICE).eval()
if hasattr(chexnet_model, 'set_grad_checkpointing'):
    chexnet_model.set_grad_checkpointing(False)

chexnet_transform = transforms.Compose([
    transforms.Resize(int(img_size * 1.14)),
    transforms.CenterCrop(img_size),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

del ckpt, cleaned_sd
gc.collect()
torch.cuda.empty_cache()
print(f'CheXNet loaded! Size={model_size}, ImgSize={img_size}')

# ======================================
# Grad-CAM: Sinh heatmap tu classification gradients
# ======================================
_gradcam_features = {}
_gradcam_grads = {}

def _save_features(module, input, output):
    _gradcam_features['value'] = output

def _save_grads(module, grad_input, grad_output):
    _gradcam_grads['value'] = grad_output[0]

hook_fwd = chexnet_model.fpn_merge.register_forward_hook(_save_features)
hook_bwd = chexnet_model.fpn_merge.register_full_backward_hook(_save_grads)
print('Grad-CAM hooks registered on fpn_merge layer.')

In [ ]:
# ======================================
# BUOC 3.2: Sinh Pseudo-Labels bang Grad-CAM
# ======================================

def gradcam_to_bboxes(gradcam_map, img_w, img_h, min_area_ratio=0.001):
    """Chuyen Grad-CAM heatmap thanh danh sach YOLO bboxes."""
    if gradcam_map.max() < 1e-6:
        return []
    
    map_resized = cv2.resize(gradcam_map, (img_w, img_h), interpolation=cv2.INTER_CUBIC)
    map_norm = (map_resized - map_resized.min()) / (map_resized.max() - map_resized.min() + 1e-8)
    map_u8 = np.clip(map_norm * 255, 0, 255).astype(np.uint8)
    
    min_area = min_area_ratio * img_w * img_h
    
    for thresh_val in [None, np.percentile(map_u8[map_u8 > 0], 50) if (map_u8 > 0).any() else 128,
                       map_u8.max() * 0.3, map_u8.max() * 0.2]:
        if thresh_val is None:
            _, binary = cv2.threshold(map_u8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        else:
            _, binary = cv2.threshold(map_u8, int(thresh_val), 255, cv2.THRESH_BINARY)
        
        contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        bboxes = []
        for cnt in contours:
            if cv2.contourArea(cnt) < min_area:
                continue
            x, y, w, h = cv2.boundingRect(cnt)
            roi = map_norm[y:y+h, x:x+w]
            mean_act = float(roi.mean()) if roi.size > 0 else 0
            bboxes.append({'x_min': x, 'y_min': y, 'x_max': x+w, 'y_max': y+h,
                           'confidence': mean_act})
        if bboxes:
            bboxes.sort(key=lambda b: b['confidence'], reverse=True)
            return bboxes[:3]
    return []


def compute_gradcam_single(image_tensor, target_class_idx):
    """Tinh Grad-CAM cho 1 anh duy nhat."""
    try:
        chexnet_model.zero_grad()
        _gradcam_features.clear()
        _gradcam_grads.clear()
        
        inp = image_tensor.unsqueeze(0).to(GPU_DEVICE)
        inp.requires_grad_(True)
        
        logits, _ = chexnet_model(inp)
        score = logits[0, target_class_idx]
        score.backward()
        
        features = _gradcam_features['value'].detach()
        grads = _gradcam_grads['value'].detach()
        
        weights = grads.mean(dim=[2, 3], keepdim=True)
        cam = (weights * features).sum(dim=1)
        cam = torch.relu(cam)
        
        c = cam[0].cpu().numpy()
        if c.max() > 0:
            c = c / c.max()
        
        del inp, logits, score, features, grads, weights, cam
        return c
    except Exception as e:
        print(f'    Grad-CAM error: {e}')
        return None
    finally:
        chexnet_model.zero_grad()
        _gradcam_features.clear()
        _gradcam_grads.clear()
        torch.cuda.empty_cache()


_debug_stats = {'total': 0, 'has_bbox': 0, 'cam_maxes': [], 'cam_means': [],
                'errors': 0}

def generate_pseudo_labels_gradcam(df, split_name):
    """Sinh pseudo-labels bang Grad-CAM."""
    global _debug_stats
    _debug_stats = {'total': 0, 'has_bbox': 0, 'cam_maxes': [], 'cam_means': [],
                    'errors': 0}
    
    nih_df = df[df['Source'] == 'NIH'].copy()
    pseudo_mask = nih_df[PSEUDO_DISEASES].max(axis=1) > 0
    target_df = nih_df[pseudo_mask].drop_duplicates(subset='Image Index')
    
    print(f'\n[{split_name}] Anh NIH can pseudo-label: {len(target_df)}')
    if len(target_df) == 0:
        return {}
    
    pseudo_labels = {}
    
    for disease in PSEUDO_DISEASES:
        chex_idx = PSEUDO_CHEXNET_IDX[disease]
        yolo_id = DISEASE_TO_YOLO_ID[disease]
        
        disease_df = target_df[target_df[disease] == 1]
        if len(disease_df) == 0:
            continue
        
        print(f'  [{disease}] {len(disease_df)} images, CheXNet idx={chex_idx}, YOLO id={yolo_id}')
        err_count = 0
        
        for _, row in tqdm(disease_df.iterrows(), total=len(disease_df),
                           desc=f'  [{disease}]', leave=False):
            filename = row['Image Index']
            # Anh da nam phang trong DATA_ROOT => ghep truc tiep
            img_path = os.path.join(DATA_ROOT, filename)
            if not os.path.exists(img_path):
                continue
            
            # Lay kich thuoc anh tu cache (nhanh) hoac doc file (fallback)
            if filename in image_size_cache:
                img_w, img_h = image_size_cache[filename]
            else:
                try:
                    with Image.open(img_path) as tmp:
                        img_w, img_h = tmp.size
                except Exception:
                    continue
            
            try:
                pil_img = Image.open(img_path).convert('RGB')
                tensor = chexnet_transform(pil_img)
                del pil_img
            except Exception:
                continue
            
            cam = compute_gradcam_single(tensor, chex_idx)
            del tensor
            
            if cam is None:
                _debug_stats['errors'] += 1
                err_count += 1
                if err_count >= 3:
                    print(f'    WARNING: {err_count} consecutive errors, skipping rest of {disease}')
                    break
                continue
            err_count = 0
            
            _debug_stats['total'] += 1
            _debug_stats['cam_maxes'].append(float(cam.max()))
            _debug_stats['cam_means'].append(float(cam.mean()))
            
            bboxes = gradcam_to_bboxes(cam, img_w, img_h)
            if bboxes:
                _debug_stats['has_bbox'] += 1
                new_boxes = [(yolo_id, b['x_min'], b['y_min'],
                              b['x_max'], b['y_max']) for b in bboxes]
                if filename in pseudo_labels:
                    pseudo_labels[filename].extend(new_boxes)
                else:
                    pseudo_labels[filename] = new_boxes
            del cam
        
        gc.collect()
        torch.cuda.empty_cache()
    
    print(f'\n[{split_name}] === GRAD-CAM STATISTICS ===')
    print(f'  Total Grad-CAM maps: {_debug_stats["total"]}')
    print(f'  Maps that produced bbox: {_debug_stats["has_bbox"]}')
    print(f'  Errors: {_debug_stats["errors"]}')
    if _debug_stats['cam_maxes']:
        maxes = np.array(_debug_stats['cam_maxes'])
        means = np.array(_debug_stats['cam_means'])
        print(f'  CAM max  - min:{maxes.min():.4f} mean:{maxes.mean():.4f} max:{maxes.max():.4f}')
        print(f'  CAM mean - min:{means.min():.6f} mean:{means.mean():.6f} max:{means.max():.6f}')
        print(f'  % maps with max > 0.1: {(maxes > 0.1).mean():.1%}')
        print(f'  % maps with max > 0.5: {(maxes > 0.5).mean():.1%}')
    
    print(f'[{split_name}] Pseudo-labels sinh duoc cho {len(pseudo_labels)} anh')
    return pseudo_labels

pseudo_train = generate_pseudo_labels_gradcam(df_train, 'Train')
pseudo_val   = generate_pseudo_labels_gradcam(df_val, 'Val')

hook_fwd.remove()
hook_bwd.remove()
del chexnet_model
gc.collect()
torch.cuda.empty_cache()
print('\nCheXNet model unloaded. GPU freed for YOLO training.')

## 5. Chuẩn bị YOLO Dataset (Symlink + Labels)

Ảnh đã nằm phẳng trong `DATA_ROOT` nhờ Cell 3 → chỉ cần:
- **Symlink** trực tiếp `DATA_ROOT/tên_ảnh` vào `images/train/` hoặc `images/val/`
- **Ghi label** `.txt` tương ứng
- Kích thước ảnh lấy từ `image_size_cache` (đã cache vào RAM)

**Nguồn labels:**
- VinDr: BBox từ CSV (qua WBF fuse)
- NIH (pseudo): BBox từ CheXNet Grad-CAM
- No Finding: File `.txt` RỖNG (negative examples)

In [ ]:
# ======================================
# BUOC 5: Tao YOLO Dataset
# ======================================
# Anh da nam PHANG trong DATA_ROOT nho Cell 3
# Kich thuoc anh da cache trong image_size_cache

images_dir = os.path.join(WORK_DIR, 'images')
labels_dir = os.path.join(WORK_DIR, 'labels')
for split in ['train', 'val']:
    os.makedirs(os.path.join(images_dir, split), exist_ok=True)
    os.makedirs(os.path.join(labels_dir, split), exist_ok=True)


def get_image_size(img_name):
    """Lay kich thuoc anh tu cache (nhanh) hoac doc file (fallback)."""
    if img_name in image_size_cache:
        return image_size_cache[img_name]
    img_path = os.path.join(DATA_ROOT, img_name)
    with Image.open(img_path) as img:
        return img.size  # (width, height)


def process_split(df, split, pseudo_labels):
    """Xu ly 1 split: symlink anh + ghi YOLO labels."""
    stats = Counter()
    grouped = df.groupby('Image Index', sort=False)
    
    for img_name, group in tqdm(grouped, desc=f'Processing {split}', total=len(grouped)):
        # Anh nam phang trong DATA_ROOT => ghep truc tiep
        img_path = os.path.join(DATA_ROOT, img_name)
        if not os.path.exists(img_path):
            stats['missing'] += 1
            continue
        
        source = group['Source'].iloc[0]
        
        # -- Tao symlink anh --
        link_path = os.path.join(images_dir, split, img_name)
        if not os.path.exists(link_path):
            os.symlink(img_path, link_path)
        
        # -- Tao label file --
        label_path = os.path.join(labels_dir, split, img_name.rsplit('.', 1)[0] + '.txt')
        lines = []
        
        if source == 'VinDr-CXR':
            has_any_bbox = group['x_min'].notna().any()
            if has_any_bbox:
                try:
                    img_w, img_h = get_image_size(img_name)
                except Exception:
                    stats['read_error'] += 1
                    continue
                
                fused = fuse_vindr_boxes(group, img_w, img_h)
                for box in fused:
                    cls_id = box['yolo_class_id']
                    x_c = np.clip(((box['x_min'] + box['x_max']) / 2) / img_w, 0, 1)
                    y_c = np.clip(((box['y_min'] + box['y_max']) / 2) / img_h, 0, 1)
                    bw = np.clip((box['x_max'] - box['x_min']) / img_w, 0, 1)
                    bh = np.clip((box['y_max'] - box['y_min']) / img_h, 0, 1)
                    lines.append(f'{cls_id} {x_c:.6f} {y_c:.6f} {bw:.6f} {bh:.6f}')
                stats['vindr_bbox'] += 1
            else:
                stats['vindr_nofinding'] += 1
        
        elif source == 'NIH':
            if img_name in pseudo_labels:
                try:
                    img_w, img_h = get_image_size(img_name)
                except Exception:
                    stats['read_error'] += 1
                    continue
                
                for (cls_id, x1, y1, x2, y2) in pseudo_labels[img_name]:
                    x_c = np.clip(((x1 + x2) / 2) / img_w, 0, 1)
                    y_c = np.clip(((y1 + y2) / 2) / img_h, 0, 1)
                    bw = np.clip((x2 - x1) / img_w, 0, 1)
                    bh = np.clip((y2 - y1) / img_h, 0, 1)
                    lines.append(f'{cls_id} {x_c:.6f} {y_c:.6f} {bw:.6f} {bh:.6f}')
                stats['nih_pseudo'] += 1
            else:
                stats['nih_nofinding'] += 1
        
        with open(label_path, 'w') as f:
            f.write('\n'.join(lines))
    
    return stats

print('=== Processing Train Split ===')
train_stats = process_split(df_train, 'train', pseudo_train)
print(f'Train stats: {dict(train_stats)}')

print('\n=== Processing Val Split ===')
val_stats = process_split(df_val, 'val', pseudo_val)
print(f'Val stats: {dict(val_stats)}')

for split in ['train', 'val']:
    label_files = glob.glob(f'{labels_dir}/{split}/*.txt')
    non_empty = sum(1 for f in label_files if os.path.getsize(f) > 0)
    print(f'\n{split}: {len(label_files)} label files ({non_empty} co bbox, '
          f'{len(label_files) - non_empty} negative)')

## 6. Tạo file `data.yaml`

14 classes đồng bộ CheXNet (bỏ No Finding).

In [ ]:
yaml_content = f"""path: {WORK_DIR}
train: images/train
val: images/val

nc: 14

names:
  0: Atelectasis
  1: Cardiomegaly
  2: Effusion
  3: Infiltration
  4: Mass
  5: Nodule
  6: Pneumonia
  7: Pneumothorax
  8: Consolidation
  9: Edema
  10: Emphysema
  11: Fibrosis
  12: Pleural_Thickening
  13: Hernia
"""

yaml_path = os.path.join(WORK_DIR, 'data.yaml')
with open(yaml_path, 'w') as f:
    f.write(yaml_content)
print(f'data.yaml saved: {yaml_path}')
print(yaml_content)

## 7. Huấn luyện Progressive (Tối ưu A100 40GB)

**Phase 1** (640px, 25 epochs): batch=**80**, workers=8, cache='ram' — Học đặc trưng cơ bản
**Phase 2** (1024px, 55 epochs): batch=**28**, workers=8, cache='ram' — Fine-tune ở độ phân giải cao

### Tối ưu cho A100:
- **Batch size lớn** — A100 40GB VRAM cho phép batch gấp 5x so với T4 16GB
- **cache='ram'** — Load toàn bộ ảnh vào RAM (83GB), bỏ qua disk I/O khi train
- **workers=8** — Tận dụng high-RAM runtime để load data nhanh hơn
- **amp=True** — A100 hỗ trợ TF32 + mixed precision native, throughput cao hơn

Augmentation cho X-quang: Tắt flipud, mosaic, mixup. Giữ fliplr, degrees, scale.

In [ ]:
# ======================================
# PHASE 1: 640px (A100 40GB optimized)
# ======================================

print('Sanitizing labels...')
sanitized = 0
removed_lines = 0
for split in ['train', 'val']:
    label_files = glob.glob(f'{WORK_DIR}/labels/{split}/*.txt')
    for lf in label_files:
        if os.path.getsize(lf) == 0:
            continue
        with open(lf, 'r') as f:
            lines = f.readlines()
        clean_lines = []
        for line in lines:
            parts = line.strip().split()
            if len(parts) != 5:
                removed_lines += 1
                continue
            cls_id, cx, cy, w, h = int(parts[0]), float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
            if w < 0.005 or h < 0.005 or w > 1.0 or h > 1.0:
                removed_lines += 1
                continue
            if cx < 0 or cx > 1 or cy < 0 or cy > 1:
                removed_lines += 1
                continue
            if cls_id < 0 or cls_id >= 14:
                removed_lines += 1
                continue
            clean_lines.append(line)
        if len(clean_lines) != len(lines):
            sanitized += 1
            with open(lf, 'w') as f:
                f.writelines(clean_lines)
print(f'Sanitized: {sanitized} files modified, {removed_lines} bad lines removed')

# --- Bat TF32 cho A100 ---
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
print('TF32 enabled for A100')

model = YOLO('yolo11m.pt')

results_p1 = model.train(
    data=yaml_path,
    epochs=25,
    imgsz=640,
    batch=80,
    workers=8,
    patience=10,
    cache='ram',
    mosaic=0.0,
    mixup=0.0,
    flipud=0.0,
    fliplr=0.5,
    degrees=5,
    scale=0.2,
    optimizer='AdamW',
    lr0=0.002,
    lrf=0.01,
    cos_lr=True,
    warmup_epochs=5,
    warmup_bias_lr=0.01,
    warmup_momentum=0.5,
    amp=True,
    name='chexnet_yolo_hybrid_phase1',
    device=DEVICE,
    exist_ok=True,
    verbose=True,
)
print('Phase 1 hoan tat!')

In [ ]:
# ======================================
# PHASE 2: 1024px (A100 40GB optimized)
# ======================================
best_phase1 = 'runs/detect/chexnet_yolo_hybrid_phase1/weights/best.pt'
if not os.path.exists(best_phase1):
    candidates = glob.glob('runs/detect/chexnet_yolo_hybrid_phase1*/weights/best.pt')
    if not candidates:
        raise FileNotFoundError('Khong tim thay model Phase 1!')
    best_phase1 = candidates[0]

print(f'Loading Phase 1 best: {best_phase1}')
model_p2 = YOLO(best_phase1)

results_p2 = model_p2.train(
    data=yaml_path,
    epochs=55,
    imgsz=1024,
    batch=28,
    workers=8,
    patience=15,
    cache='ram',
    mosaic=0.0,
    mixup=0.0,
    flipud=0.0,
    fliplr=0.5,
    degrees=10,
    scale=0.3,
    optimizer='AdamW',
    cos_lr=True,
    lr0=0.001,
    lrf=0.01,
    warmup_epochs=3,
    amp=True,
    name='chexnet_yolo_hybrid_phase2',
    device=DEVICE,
    exist_ok=True,
    verbose=True,
)
print('Phase 2 hoan tat!')

In [ ]:
# ======================================
# RESUME - Chay cell nay neu training bi gian doan
# ======================================
# Bo comment khi can resume:

# from ultralytics import YOLO
# model_resume = YOLO('runs/detect/chexnet_yolo_hybrid_phase2/weights/last.pt')
# model_resume.train(resume=True)

## 8. Đánh giá với TTA + WBF

Dự đoán ảnh gốc + flip ngang, gộp bằng WBF.

In [ ]:
def tta_wbf_predict(model, img_path, imgsz=1024, conf=0.01, iou_thr=0.4):
    """TTA inference: anh goc + flip ngang, gop bang WBF."""
    img = cv2.imread(img_path)
    img_flip = cv2.flip(img, 1)
    
    res_orig = model.predict(img, imgsz=imgsz, conf=conf, verbose=False)[0]
    res_flip = model.predict(img_flip, imgsz=imgsz, conf=conf, verbose=False)[0]
    
    b1 = res_orig.boxes.xyxyn.cpu().numpy().tolist() if len(res_orig.boxes) > 0 else []
    s1 = res_orig.boxes.conf.cpu().numpy().tolist() if len(res_orig.boxes) > 0 else []
    l1 = res_orig.boxes.cls.cpu().numpy().astype(int).tolist() if len(res_orig.boxes) > 0 else []
    
    if len(res_flip.boxes) > 0:
        b2_raw = res_flip.boxes.xyxyn.cpu().numpy().copy()
        b2_raw[:, [0, 2]] = 1.0 - b2_raw[:, [2, 0]]
        b2, s2, l2 = b2_raw.tolist(), res_flip.boxes.conf.cpu().numpy().tolist(), res_flip.boxes.cls.cpu().numpy().astype(int).tolist()
    else:
        b2, s2, l2 = [], [], []
    
    if not b1 and not b2:
        return np.array([]), np.array([]), np.array([])
    
    all_boxes = [b1 if b1 else [[0,0,0,0]], b2 if b2 else [[0,0,0,0]]]
    all_scores = [s1 if s1 else [0], s2 if s2 else [0]]
    all_labels = [l1 if l1 else [0], l2 if l2 else [0]]
    
    fused_boxes, fused_scores, fused_labels = weighted_boxes_fusion(
        all_boxes, all_scores, all_labels, weights=[1, 1],
        iou_thr=iou_thr, skip_box_thr=0.01
    )
    return fused_boxes, fused_scores, fused_labels

# -- Danh gia --
best_p2 = 'runs/detect/chexnet_yolo_hybrid_phase2/weights/best.pt'
if os.path.exists(best_p2):
    best_model = YOLO(best_p2)
    print('Standard evaluation:')
    metrics = best_model.val(data=yaml_path, imgsz=1024)
    print(f'mAP50: {metrics.box.map50:.4f}')
    print(f'mAP50-95: {metrics.box.map:.4f}')
    print('\nPer-class AP50:')
    for i, name in enumerate(YOLO_CLASSES):
        if i < len(metrics.box.ap50):
            print(f'  {name:25s}: {metrics.box.ap50[i]:.4f}')
else:
    print('Chua co model Phase 2.')

## 9. Cascade 2 giai đoạn + Mapping

`Final_Score = P_CheXNet_Classifier * Conf_YOLO_Detector`

YOLO class ID da dong bo 1-1 voi CheXNet (YOLO_id + 1 = CheXNet_id).

In [ ]:
def cascade_filter(yolo_boxes, yolo_confs, yolo_classes, chexnet_probs):
    """Loc YOLO boxes bang xac suat tu CheXNet classifier."""
    CLINICAL_THRESHOLDS = {
        6:  0.05,  # Pneumonia
        7:  0.05,  # Pneumothorax
        1:  0.10,  # Cardiomegaly
        2:  0.10,  # Effusion
        8:  0.12,  # Consolidation
    }
    DEFAULT_THRESHOLD = 0.15
    
    filtered = {'boxes': [], 'confs': [], 'classes': []}
    for box, conf, cls_id in zip(yolo_boxes, yolo_confs, yolo_classes):
        cls_id = int(cls_id)
        p_class = chexnet_probs.get(cls_id, 0.5)
        final_score = float(conf) * p_class
        thresh = CLINICAL_THRESHOLDS.get(cls_id, DEFAULT_THRESHOLD)
        if final_score >= thresh:
            filtered['boxes'].append(box)
            filtered['confs'].append(final_score)
            filtered['classes'].append(cls_id)
    return filtered

YOLO_TO_CHEXNET_IDX = {i: i + 1 for i in range(14)}
print('YOLO <-> CheXNet mapping:')
for yolo_id, chex_id in YOLO_TO_CHEXNET_IDX.items():
    print(f'  YOLO {yolo_id:2d} ({YOLO_CLASSES[yolo_id]:25s}) <-> CheXNet {chex_id:2d} ({CHEXNET_CLASS_NAMES[chex_id]})')

In [ ]:
# ======================================
# Export model tot nhat
# ======================================
best_paths = [
    'runs/detect/chexnet_yolo_hybrid_phase2/weights/best.pt',
    'runs/detect/chexnet_yolo_hybrid_phase1/weights/best.pt',
]

exported = False
for bp in best_paths:
    if os.path.exists(bp):
        out_path = '/content/yolo11m_hybrid.pt'
        shutil.copy(bp, out_path)
        print(f'Model exported: {bp} -> {out_path}')
        m = YOLO(out_path)
        m.info()
        exported = True
        break

if not exported:
    print('Chua co model trained.')

if exported:
    drive_out = '/content/drive/MyDrive/CheXNet/yolo11m_hybrid.pt'
    os.makedirs(os.path.dirname(drive_out), exist_ok=True)
    shutil.copy(out_path, drive_out)
    print(f'Model saved to Drive: {drive_out}')

## 10. Tổng kết

### Đã thực hiện:
- Knowledge Distillation: CheXNet Grad-CAM sinh Pseudo BBox (Pneumonia, Edema, Emphysema, Hernia)
- Dataset Hybrid: VinDr-CXR GT + NIH Pseudo-labels, 14 classes đồng bộ CheXNet
- WBF trên Ground Truth (gộp nhiều bác sĩ thành consensus box)
- Ảnh gom phẳng vào 1 thư mục → truy cập trực tiếp, không cần tìm kiếm
- Symlink tiết kiệm disk (không copy 50GB ảnh)
- Progressive Training (640px rồi 1024px)
- TTA + WBF Inference
- Cascade 2 giai đoạn (CheXNet x YOLO)
- YOLO - CheXNet mapping đồng bộ (class name 1-1)

### Tối ưu A100 40GB:
- TF32 enabled (throughput ~3x so với FP32)
- Phase 1: batch 80, cache='ram'
- Phase 2: batch 28, cache='ram'
- Workers 8 (tận dụng high-RAM)
- Cache kích thước ảnh vào RAM (bỏ qua disk I/O khi tạo labels)

### Bước tiếp theo:
1. Tải file `yolo11m_hybrid.pt` về máy (đã tự động lưu lên Drive)
2. Đặt vào `CheXNet/Trainedmodel/yolov11m.pt`
3. Cập nhật `Backend/yolo_detector.py` với mapping 14 class mới
4. Cập nhật `Backend/main.py` để render heatmap BÊN TRONG YOLO bbox